In [3]:
# ================================================================
# SYNAPSEPM - REAL TRAINED MODEL EVALUATION
# Single-cell evaluation / diagnostics
#
# Model:
# C:\Users\DELL\OneDrive\Desktop\Synapse PM\trained_model.zip
#
# Dataset:
# C:\Users\DELL\OneDrive\Desktop\Synapse PM\data\healthcare_iot_target_dataset.csv
#
# This code:
#   1. Extracts the trained model ZIP
#   2. Inspects ALL model artifacts
#   3. Detects XGBoost / sklearn / PyTorch / Keras artifacts
#   4. Reads the dataset
#   5. Determines the model's REAL expected feature count
#   6. Attempts to recover feature names from the trained model
#   7. Looks for a legitimate target column
#   8. NEVER creates fake targets
#   9. NEVER randomly selects 10 columns for a 26-feature model
#  10. Evaluates only if feature/target alignment is valid
# ================================================================

import os
import sys
import json
import zipfile
import pickle
import warnings
import traceback
import shutil
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# OPTIONAL LIBRARIES
# ---------------------------------------------------------------

print("=" * 80)
print("SYNAPSEPM - REAL MODEL EVALUATION")
print("=" * 80)

MODEL_ZIP = Path(
    r"C:\Users\DELL\OneDrive\Desktop\Synapse PM\trained_model.zip"
)

DATASET_PATH = Path(
    r"C:\Users\DELL\OneDrive\Desktop\Synapse PM\data\healthcare_iot_target_dataset.csv"
)

WORK_DIR = MODEL_ZIP.parent / "_synapsepm_model_eval"

# ---------------------------------------------------------------
# CHECK FILES
# ---------------------------------------------------------------

print("\n[1/12] Checking files...")

if not MODEL_ZIP.exists():
    raise FileNotFoundError(
        f"\nModel ZIP was not found:\n{MODEL_ZIP}"
    )

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"\nDataset was not found:\n{DATASET_PATH}"
    )

print("✓ Model ZIP:", MODEL_ZIP)
print("✓ Dataset:", DATASET_PATH)

# ---------------------------------------------------------------
# EXTRACT MODEL
# ---------------------------------------------------------------

print("\n[2/12] Extracting trained model...")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

WORK_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(MODEL_ZIP, "r") as z:
    z.extractall(WORK_DIR)

print("✓ Model extracted to:")
print(WORK_DIR)

# ---------------------------------------------------------------
# SHOW ZIP CONTENTS
# ---------------------------------------------------------------

print("\n[3/12] Model artifacts found:")

all_files = []

for p in WORK_DIR.rglob("*"):
    if p.is_file():
        all_files.append(p)

if not all_files:
    raise RuntimeError("The ZIP file contains no files.")

for p in all_files:
    try:
        size_mb = p.stat().st_size / (1024 * 1024)
    except:
        size_mb = 0

    print(
        f"  - {p.relative_to(WORK_DIR)} "
        f"({size_mb:.2f} MB)"
    )

# ---------------------------------------------------------------
# LOAD DATASET
# ---------------------------------------------------------------

print("\n[4/12] Loading evaluation dataset...")

df = pd.read_csv(DATASET_PATH)

print("✓ Dataset loaded")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDataset columns:")
for i, c in enumerate(df.columns):
    print(f"  [{i:02d}] {c}")

# ---------------------------------------------------------------
# BASIC DATASET INFORMATION
# ---------------------------------------------------------------

print("\nDataset shape:", df.shape)

print("\nData types:")
print(df.dtypes.to_string())

print("\nMissing values:")
missing = df.isna().sum()
missing = missing[missing > 0]

if len(missing):
    print(missing.to_string())
else:
    print("✓ No missing values")

# ---------------------------------------------------------------
# FIND MODEL FILES
# ---------------------------------------------------------------

print("\n[5/12] Searching for trained model artifacts...")

model_candidates = []
scaler_candidates = []
metadata_candidates = []

for p in all_files:

    name = p.name.lower()

    if name.endswith((
        ".pkl",
        ".pickle",
        ".joblib",
        ".h5",
        ".keras",
        ".pt",
        ".pth",
        ".onnx",
        ".json",
        ".ubj"
    )):
        model_candidates.append(p)

    if any(x in name for x in [
        "scaler",
        "transform",
        "preprocess",
        "normalizer"
    ]):
        scaler_candidates.append(p)

    if any(x in name for x in [
        "metadata",
        "config",
        "feature",
        "schema"
    ]):
        metadata_candidates.append(p)

print("\nPotential model files:")

for p in model_candidates:
    print(" ", p.relative_to(WORK_DIR))

print("\nPotential scaler files:")

for p in scaler_candidates:
    print(" ", p.relative_to(WORK_DIR))

print("\nPotential metadata files:")

for p in metadata_candidates:
    print(" ", p.relative_to(WORK_DIR))

# ---------------------------------------------------------------
# LOAD POSSIBLE METADATA
# ---------------------------------------------------------------

print("\n[6/12] Inspecting metadata...")

metadata_objects = []

for p in metadata_candidates:

    if p.suffix.lower() != ".json":
        continue

    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)

        metadata_objects.append((p, obj))

        print("\nMetadata:", p.name)
        print(json.dumps(obj, indent=2)[:10000])

    except Exception as e:
        print(
            f"Could not read {p.name}: {e}"
        )

# ---------------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------------

print("\n[7/12] Attempting to load trained model...")

model = None
model_path = None
model_type = None

# First try XGBoost
try:

    import xgboost as xgb

    xgb_files = [
        p for p in all_files
        if p.suffix.lower() in [".json", ".ubj"]
    ]

    for p in xgb_files:

        try:

            booster = xgb.XGBModel()

            booster.load_model(str(p))

            model = booster
            model_path = p
            model_type = "xgboost"

            print("✓ Loaded XGBoost model:")
            print(p)

            break

        except Exception:
            pass

except Exception as e:
    print("XGBoost unavailable:", e)

# ---------------------------------------------------------------
# TRY PICKLE / JOBLIB
# ---------------------------------------------------------------

if model is None:

    pickle_files = [
        p for p in all_files
        if p.suffix.lower() in [
            ".pkl",
            ".pickle",
            ".joblib"
        ]
    ]

    for p in pickle_files:

        try:

            obj = None

            if p.suffix.lower() == ".joblib":
                import joblib
                obj = joblib.load(p)

            else:
                with open(p, "rb") as f:
                    obj = pickle.load(f)

            # Check whether object looks like a model
            if hasattr(obj, "predict"):

                model = obj
                model_path = p
                model_type = "sklearn-compatible"

                print("✓ Loaded sklearn-compatible model:")
                print(p)

                print("Model class:")
                print(type(model))

                break

        except Exception as e:
            print(
                f"Could not load {p.name}: {str(e)[:150]}"
            )

# ---------------------------------------------------------------
# TRY PYTORCH
# ---------------------------------------------------------------

if model is None:

    pt_files = [
        p for p in all_files
        if p.suffix.lower() in [".pt", ".pth"]
    ]

    if pt_files:

        try:

            import torch

            for p in pt_files:

                try:

                    obj = torch.load(
                        p,
                        map_location="cpu",
                        weights_only=False
                    )

                    model = obj
                    model_path = p
                    model_type = "pytorch"

                    print("✓ Loaded PyTorch artifact:")
                    print(p)

                    print("Object type:")
                    print(type(model))

                    break

                except Exception as e:
                    print(
                        f"PyTorch load failed for {p.name}: "
                        f"{str(e)[:200]}"
                    )

        except Exception as e:
            print("PyTorch unavailable:", e)

# ---------------------------------------------------------------
# MODEL FOUND?
# ---------------------------------------------------------------

if model is None:

    print("\n" + "=" * 80)
    print("MODEL COULD NOT BE LOADED")
    print("=" * 80)

    print("""
The ZIP contains artifacts, but none could be automatically loaded
as a prediction model.

The artifact listing above is important.

If your ZIP contains a custom model class, the exact model-loading
code used during training is required.
""")

    raise RuntimeError(
        "No compatible trained model could be loaded."
    )

print("\nModel type:", model_type)
print("Model path:", model_path)

# ---------------------------------------------------------------
# DETERMINE EXPECTED FEATURE COUNT
# ---------------------------------------------------------------

print("\n[8/12] Determining model input requirements...")

expected_features = None
feature_names = None

# XGBoost
if model_type == "xgboost":

    try:
        expected_features = model.get_booster().num_features()
    except:
        try:
            expected_features = model.n_features_in_
        except:
            pass

    try:
        feature_names = model.get_booster().feature_names
    except:
        pass

# sklearn
if expected_features is None:

    if hasattr(model, "n_features_in_"):
        try:
            expected_features = int(
                model.n_features_in_
            )
        except:
            pass

if feature_names is None:

    if hasattr(model, "feature_names_in_"):

        try:
            feature_names = list(
                model.feature_names_in_
            )

        except:
            pass

print("\nExpected feature count:")

if expected_features is not None:
    print("✓", expected_features)

else:
    print("UNKNOWN")

print("\nFeature names from model:")

if feature_names:

    for i, f in enumerate(feature_names):
        print(f"  [{i:02d}] {f}")

else:
    print("No explicit feature names stored in model.")

# ---------------------------------------------------------------
# SEARCH METADATA FOR FEATURES/TARGET
# ---------------------------------------------------------------

print("\nSearching metadata for feature/target definitions...")

metadata_features = None
metadata_target = None

def recursively_find(obj, keys):
    """
    Recursively search JSON/dict/list structures.
    """
    if isinstance(obj, dict):

        for k, v in obj.items():

            key = str(k).lower()

            if key in keys:
                return v

            found = recursively_find(v, keys)

            if found is not None:
                return found

    elif isinstance(obj, list):

        for item in obj:

            found = recursively_find(
                item,
                keys
            )

            if found is not None:
                return found

    return None


for p, obj in metadata_objects:

    if metadata_features is None:

        metadata_features = recursively_find(
            obj,
            {
                "features",
                "feature_names",
                "input_features",
                "feature_columns",
                "columns"
            }
        )

    if metadata_target is None:

        metadata_target = recursively_find(
            obj,
            {
                "target",
                "target_column",
                "target_name",
                "label",
                "label_column",
                "y_column",
                "output",
                "output_column"
            }
        )

print("\nMetadata feature definition:")
print(metadata_features)

print("\nMetadata target definition:")
print(metadata_target)

# ---------------------------------------------------------------
# RESOLVE FEATURES
# ---------------------------------------------------------------

if feature_names is None and metadata_features is not None:

    if isinstance(metadata_features, list):

        feature_names = [
            str(x)
            for x in metadata_features
        ]

if feature_names:

    feature_names = [
        str(x)
        for x in feature_names
        if str(x) in df.columns
    ]

# ---------------------------------------------------------------
# HANDLE XGBOOST WITHOUT FEATURE NAMES
# ---------------------------------------------------------------

if expected_features is not None:

    print("\nModel requires:", expected_features, "features")

if feature_names:

    print(
        "Matching model feature names found in dataset:",
        len(feature_names)
    )

# ---------------------------------------------------------------
# TARGET DETECTION
# ---------------------------------------------------------------

print("\n[9/12] Determining REAL evaluation target...")

target_column = None

# 1. Metadata target
if metadata_target:

    if isinstance(metadata_target, str):

        if metadata_target in df.columns:
            target_column = metadata_target

# 2. Common exact target names
if target_column is None:

    exact_candidates = [
        "RUL",
        "rul",
        "RemainingUsefulLife",
        "remaining_useful_life",
        "remaining_useful_life_days",
        "remaining_life",
        "failure",
        "Failure",
        "machine_failure",
        "MachineFailure",
        "target",
        "Target",
        "label",
        "Label",
        "y",
        "Y"
    ]

    for c in exact_candidates:

        if c in df.columns:
            target_column = c
            break

# 3. Name-based candidates
if target_column is None:

    target_candidates = []

    for c in df.columns:

        lc = str(c).strip().lower()

        score = 0

        if "rul" in lc:
            score += 100

        if "remaining" in lc and "life" in lc:
            score += 100

        if "failure" in lc:
            score += 90

        if "target" in lc:
            score += 80

        if "label" in lc:
            score += 70

        if lc in ["y", "output"]:
            score += 50

        if score > 0:
            target_candidates.append(
                (score, c)
            )

    target_candidates.sort(
        reverse=True
    )

    if target_candidates:

        # Only automatically select if clearly dominant
        if len(target_candidates) == 1:

            target_column = target_candidates[0][1]

        elif (
            target_candidates[0][0]
            > target_candidates[1][0]
        ):

            target_column = target_candidates[0][1]

# ---------------------------------------------------------------
# TARGET RESULT
# ---------------------------------------------------------------

print("\nTarget candidates:")

for c in df.columns:

    lc = str(c).lower()

    if any(k in lc for k in [
        "rul",
        "remaining",
        "failure",
        "target",
        "label",
        "output"
    ]):

        print("  ->", c)

if target_column:

    print(
        "\n✓ REAL TARGET SELECTED:",
        target_column
    )

else:

    print("""
    
NO SAFE TARGET COLUMN WAS FOUND.

This is NOT a model failure.

It means the CSV does not clearly contain the ground-truth
variable needed to calculate accuracy/error.

The evaluation cannot honestly continue without a real target.

Possible target-like columns in your dataset are listed above.
""")

    print("\nALL DATASET COLUMNS:")

    for c in df.columns:
        print(" ", c)

    raise RuntimeError(
        """
No safe ground-truth target column could be identified.

I will NOT invent a target or calculate fake accuracy.
Check the target column used during training.
"""
    )

# ---------------------------------------------------------------
# FEATURE ALIGNMENT
# ---------------------------------------------------------------

print("\n[10/12] Aligning evaluation features...")

if feature_names:

    missing_features = [
        f for f in feature_names
        if f not in df.columns
    ]

    if missing_features:

        print("\nMISSING FEATURES:")

        for f in missing_features:
            print("  -", f)

        raise RuntimeError(
            "\nThe evaluation dataset does not contain all "
            "features used by the trained model."
        )

    X_eval = df[feature_names].copy()

else:

    # If the model has no feature names, ONLY allow this
    # when the dataset has exactly expected_features
    # after removing the target.

    candidate_features = [
        c for c in df.columns
        if c != target_column
    ]

    if expected_features is None:

        raise RuntimeError(
            """
The model does not expose its expected feature count
or feature names.

The evaluation cannot safely construct X.
"""
        )

    if len(candidate_features) != expected_features:

        print(
            "\nDataset feature count:",
            len(candidate_features)
        )

        print(
            "Model expected:",
            expected_features
        )

        raise RuntimeError(
            f"""
FEATURE MISMATCH.

Model expects {expected_features} input features.

Dataset has {len(candidate_features)} possible input
columns after removing target '{target_column}'.

I will NOT randomly select {expected_features} columns.
"""
        )

    X_eval = df[candidate_features].copy()

y_true = df[target_column].copy()

print(
    "\n✓ X shape:",
    X_eval.shape
)

print(
    "✓ y shape:",
    y_true.shape
)

# ---------------------------------------------------------------
# CLEAN NUMERIC FEATURES
# ---------------------------------------------------------------

print("\nCleaning feature matrix...")

# Convert boolean to integer
for c in X_eval.columns:

    if X_eval[c].dtype == bool:
        X_eval[c] = X_eval[c].astype(int)

# Convert numeric-looking strings
for c in X_eval.columns:

    if X_eval[c].dtype == object:

        converted = pd.to_numeric(
            X_eval[c],
            errors="coerce"
        )

        # Only replace if conversion is mostly successful
        valid_ratio = converted.notna().mean()

        if valid_ratio >= 0.95:
            X_eval[c] = converted

# Check nonnumeric
non_numeric = X_eval.select_dtypes(
    exclude=[np.number]
).columns.tolist()

if non_numeric:

    print("\nNon-numeric features detected:")

    for c in non_numeric:
        print(" ", c)

    raise RuntimeError(
        """
The model expects numeric features, but the evaluation
dataset contains non-numeric columns.

The preprocessing used during training is required.
"""
    )

# ---------------------------------------------------------------
# HANDLE MISSING VALUES
# ---------------------------------------------------------------

if X_eval.isna().any().any():

    print(
        "\nWARNING: Missing values exist in X."
    )

    print(
        X_eval.isna().sum()[
            X_eval.isna().sum() > 0
        ]
    )

    # Only use median if the model itself supports missing values.
    # XGBoost does.
    if model_type == "xgboost":

        print(
            "XGBoost detected; preserving NaN values."
        )

    else:

        # For sklearn models, use median ONLY as a transparent
        # evaluation preprocessing fallback.
        print(
            "Applying median imputation for evaluation."
        )

        X_eval = X_eval.fillna(
            X_eval.median(numeric_only=True)
        )

# ---------------------------------------------------------------
# TARGET CLEANING
# ---------------------------------------------------------------

y_numeric = pd.to_numeric(
    y_true,
    errors="coerce"
)

valid_mask = y_numeric.notna()

if valid_mask.sum() == 0:

    raise RuntimeError(
        f"""
Target '{target_column}' is not numeric.

This appears to be a classification target.

The script will now determine classification metrics.
"""
    )

y_true = y_numeric.values

X_eval = X_eval.loc[
    valid_mask
].reset_index(drop=True)

# ---------------------------------------------------------------
# PREDICTION
# ---------------------------------------------------------------

print("\n[11/12] Running REAL model predictions...")

print(
    "Number of evaluation samples:",
    len(X_eval)
)

try:

    predictions = model.predict(
        X_eval
    )

    predictions = np.asarray(
        predictions
    ).reshape(-1)

except Exception as e:

    print("\nMODEL PREDICTION FAILED")
    print("=" * 80)
    traceback.print_exc()

    raise RuntimeError(
        """
The trained model cannot accept the prepared dataset.

This usually means the preprocessing used during training
is missing or the feature order/types differ.
"""
    )

print("\n✓ Prediction completed")

print(
    "Prediction count:",
    len(predictions)
)

# ---------------------------------------------------------------
# SHAPE CHECK
# ---------------------------------------------------------------

if len(predictions) != len(y_true):

    raise RuntimeError(
        f"""
Prediction count mismatch.

Predictions: {len(predictions)}
Ground truth: {len(y_true)}
"""
    )

# ---------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------

print("\n[12/12] Calculating REAL metrics...")

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score
)

mae = mean_absolute_error(
    y_true,
    predictions
)

mse = mean_squared_error(
    y_true,
    predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_true,
    predictions
)

evs = explained_variance_score(
    y_true,
    predictions
)

# MAPE with protection against zero
nonzero = np.abs(y_true) > 1e-12

if nonzero.sum() > 0:

    mape = (
        np.mean(
            np.abs(
                (
                    y_true[nonzero]
                    - predictions[nonzero]
                )
                /
                y_true[nonzero]
            )
        )
        * 100
    )

else:

    mape = np.nan

# ---------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------

print("\n")
print("=" * 80)
print("                 SYNAPSEPM MODEL RESULTS")
print("=" * 80)

print(
    f"\nModel type             : {model_type}"
)

print(
    f"Model file             : {model_path.name}"
)

print(
    f"Evaluation dataset     : {DATASET_PATH.name}"
)

print(
    f"Evaluation samples     : {len(y_true):,}"
)

print(
    f"Input features         : {X_eval.shape[1]}"
)

print(
    f"Ground-truth target    : {target_column}"
)

print("\nREGRESSION METRICS")
print("-" * 50)

print(
    f"MAE                    : {mae:.6f}"
)

print(
    f"RMSE                   : {rmse:.6f}"
)

print(
    f"MSE                    : {mse:.6f}"
)

print(
    f"R²                     : {r2:.6f}"
)

print(
    f"Explained Variance     : {evs:.6f}"
)

if not np.isnan(mape):

    print(
        f"MAPE                   : {mape:.2f}%"
    )

print("\nTARGET STATISTICS")
print("-" * 50)

print(
    f"Actual mean            : {np.mean(y_true):.6f}"
)

print(
    f"Prediction mean        : {np.mean(predictions):.6f}"
)

print(
    f"Actual std             : {np.std(y_true):.6f}"
)

print(
    f"Prediction std         : {np.std(predictions):.6f}"
)

print(
    f"Actual min             : {np.min(y_true):.6f}"
)

print(
    f"Actual max             : {np.max(y_true):.6f}"
)

print(
    f"Prediction min         : {np.min(predictions):.6f}"
)

print(
    f"Prediction max         : {np.max(predictions):.6f}"
)

# ---------------------------------------------------------------
# ERROR ANALYSIS
# ---------------------------------------------------------------

errors = y_true - predictions
absolute_errors = np.abs(errors)

print("\nERROR ANALYSIS")
print("-" * 50)

print(
    f"Mean error             : {np.mean(errors):.6f}"
)

print(
    f"Median absolute error  : "
    f"{np.median(absolute_errors):.6f}"
)

print(
    f"90th percentile error  : "
    f"{np.percentile(absolute_errors, 90):.6f}"
)

print(
    f"95th percentile error  : "
    f"{np.percentile(absolute_errors, 95):.6f}"
)

print(
    f"Maximum absolute error : "
    f"{np.max(absolute_errors):.6f}"
)

# ---------------------------------------------------------------
# ACCURACY WITHIN ERROR THRESHOLDS
# ---------------------------------------------------------------

print("\nPREDICTION ACCURACY BY ERROR TOLERANCE")
print("-" * 50)

for threshold in [
    0.5,
    1,
    2,
    5,
    10
]:

    percentage = (
        np.mean(
            absolute_errors <= threshold
        )
        * 100
    )

    print(
        f"Within ±{threshold:<5} : "
        f"{percentage:>7.2f}%"
    )

# ---------------------------------------------------------------
# SAMPLE PREDICTIONS
# ---------------------------------------------------------------

results = pd.DataFrame({

    "Actual": y_true,

    "Predicted": predictions,

    "Error": errors,

    "Absolute_Error": absolute_errors

})

print("\nSAMPLE REAL PREDICTIONS")
print("-" * 80)

print(
    results.head(20).to_string(
        index=False
    )
)

# ---------------------------------------------------------------
# SAVE RESULTS
# ---------------------------------------------------------------

output_file = (
    MODEL_ZIP.parent
    / "synapsepm_real_model_evaluation.csv"
)

results.to_csv(
    output_file,
    index=False
)

print("\n")
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(
    "\nResults saved to:"
)

print(output_file)

print("\nIMPORTANT:")

print("""
These metrics are calculated from:

    trained model
        +
    actual dataset features
        +
    actual dataset target

No random target was generated.
No fake predictions were generated.
No random feature selection was performed.
""")

# ---------------------------------------------------------------
# OPTIONAL PLOTS
# ---------------------------------------------------------------

import matplotlib.pyplot as plt

plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    y_true,
    predictions,
    alpha=0.5
)

minimum = min(
    np.min(y_true),
    np.min(predictions)
)

maximum = max(
    np.max(y_true),
    np.max(predictions)
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel(
    "Actual"
)

plt.ylabel(
    "Predicted"
)

plt.title(
    "SynapsePM - Actual vs Predicted"
)

plt.grid(
    alpha=0.2
)

plt.tight_layout()

plt.show()

# ---------------------------------------------------------------

plt.figure(
    figsize=(8, 6)
)

plt.hist(
    errors,
    bins=50
)

plt.xlabel(
    "Prediction Error"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "SynapsePM - Prediction Error Distribution"
)

plt.grid(
    alpha=0.2
)

plt.tight_layout()

plt.show()

SYNAPSEPM - REAL MODEL EVALUATION

[1/12] Checking files...
✓ Model ZIP: C:\Users\DELL\OneDrive\Desktop\Synapse PM\trained_model.zip
✓ Dataset: C:\Users\DELL\OneDrive\Desktop\Synapse PM\data\healthcare_iot_target_dataset.csv

[2/12] Extracting trained model...
✓ Model extracted to:
C:\Users\DELL\OneDrive\Desktop\Synapse PM\_synapsepm_model_eval

[3/12] Model artifacts found:
  - reports\training_metrics.csv (0.02 MB)
  - reports\training_summary.json (0.07 MB)
  - ensembles\datasets_aps_failure_test_set_ensemble\ensemble.pkl (17.48 MB)
  - ensembles\datasets_aps_failure_test_set_processed_8bit_ensemble\ensemble.pkl (16.59 MB)
  - ensembles\datasets_aps_failure_training_set_ensemble\ensemble.pkl (20.08 MB)
  - ensembles\datasets_aps_failure_training_set_processed_8bit_ensemble\ensemble.pkl (18.51 MB)
  - ensembles\datasets_dataset_ensemble\ensemble.pkl (9.12 MB)
  - ensembles\datasets_data_ensemble\ensemble.pkl (8.93 MB)
  - ensembles\datasets_data_fixed_ensemble\ensemble.pkl (38.22 MB)

RuntimeError: 
No safe ground-truth target column could be identified.

I will NOT invent a target or calculate fake accuracy.
Check the target column used during training.


In [ ]:
#!/usr/bin/env python3
"""
SYNAPSEPM - Model Evaluation & Testing
Simple, ready-to-run evaluation script
"""

import os
import sys
import json
import zipfile
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. EXTRACT AND LOAD MODELS
# ============================================================

print("=" * 80)
print("  🧠 SYNAPSEPM - MODEL EVALUATION")
print("=" * 80)

# Path to your trained model zip
MODEL_ZIP = r"C:\Users\DELL\OneDrive\Desktop\Synapse PM\trained_model.zip"
EXTRACT_DIR = Path("./extracted_models")
EXTRACT_DIR.mkdir(exist_ok=True)

print(f"\n📦 Extracting models from: {MODEL_ZIP}")

try:
    with zipfile.ZipFile(MODEL_ZIP, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f"✅ Models extracted to: {EXTRACT_DIR}")
except FileNotFoundError:
    print(f"❌ Model zip not found at: {MODEL_ZIP}")
    print("   Please update MODEL_ZIP path")
    sys.exit(1)
except Exception as e:
    print(f"❌ Failed to extract: {e}")
    sys.exit(1)

# ============================================================
# 2. LOAD MODELS
# ============================================================

print("\n📊 Loading models...")

models = {}
scalers = {}
feature_names = {}

# Find all model files
model_files = list(EXTRACT_DIR.glob("*.pkl")) + list(EXTRACT_DIR.glob("*.pth"))
scaler_files = list(EXTRACT_DIR.glob("*scaler.pkl"))

print(f"   Found {len(model_files)} model files")
print(f"   Found {len(scaler_files)} scaler files")

# Load models
for file in model_files:
    try:
        if file.suffix == '.pkl':
            model = joblib.load(file)
            model_name = file.stem.replace('_model', '')
            models[model_name] = model
            print(f"   ✅ Loaded: {file.name}")
        elif file.suffix == '.pth':
            import torch
            # Try to load as state dict
            model = torch.load(file, map_location='cpu')
            model_name = file.stem.replace('_model', '')
            models[model_name] = model
            print(f"   ✅ Loaded: {file.name}")
    except Exception as e:
        print(f"   ❌ Failed to load {file.name}: {e}")

# Load scalers
for file in scaler_files:
    try:
        scaler = joblib.load(file)
        scaler_name = file.stem.replace('_scaler', '')
        scalers[scaler_name] = scaler
        print(f"   ✅ Loaded scaler: {file.name}")
    except Exception as e:
        print(f"   ❌ Failed to load scaler {file.name}: {e}")

# Load feature names if available
feature_file = EXTRACT_DIR / "feature_names.json"
if feature_file.exists():
    with open(feature_file, 'r') as f:
        feature_names = json.load(f)
    print(f"   ✅ Loaded feature names: {len(feature_names)} features")

# Load model info
info_file = EXTRACT_DIR / "model_info.json"
if info_file.exists():
    with open(info_file, 'r') as f:
        model_info = json.load(f)
    print(f"   ✅ Loaded model info")
    for key, value in model_info.items():
        if key != 'feature_names':
            print(f"      {key}: {value}")

print(f"\n✅ Loaded {len(models)} models")

# ============================================================
# 3. LOAD TEST DATA
# ============================================================

print("\n" + "=" * 80)
print("  📊 LOAD TEST DATA")
print("=" * 80)

# Update this path to your test data
TEST_DATA_PATH = input("\n📁 Enter path to your test data (CSV file): ").strip()

if not TEST_DATA_PATH or not Path(TEST_DATA_PATH).exists():
    print("\n⚠️ No valid test data path provided.")
    print("   Creating synthetic test data for demonstration...")
    
    # Create synthetic test data
    n_samples = 100
    n_features = 25
    
    np.random.seed(42)
    X_test = np.random.randn(n_samples, n_features) * 10 + 50
    y_test = np.random.exponential(500, n_samples) + 100
    
    print(f"   Generated {n_samples} synthetic samples with {n_features} features")
else:
    print(f"📊 Loading test data from: {TEST_DATA_PATH}")
    try:
        df = pd.read_csv(TEST_DATA_PATH)
        print(f"   Loaded {len(df)} rows, {len(df.columns)} columns")
        
        # Find target column (RUL)
        target_cols = ['rul', 'RUL', 'remaining_useful_life', 'target', 'y']
        target_col = None
        for col in target_cols:
            if col in df.columns:
                target_col = col
                break
        
        if target_col is None:
            print("   ⚠️ No target column found. Using last column as target.")
            target_col = df.columns[-1]
        
        feature_cols = [c for c in df.columns if c != target_col]
        X_test = df[feature_cols].values
        y_test = df[target_col].values
        
        print(f"   Features: {len(feature_cols)}")
        print(f"   Target: {target_col}")
        
    except Exception as e:
        print(f"❌ Failed to load test data: {e}")
        sys.exit(1)

# ============================================================
# 4. PREPROCESS TEST DATA
# ============================================================

print("\n🔧 Preprocessing test data...")

# Use the first available scaler
scaler = None
scaler_name = None
for name, sc in scalers.items():
    scaler = sc
    scaler_name = name
    break

if scaler is not None:
    try:
        X_test_scaled = scaler.transform(X_test)
        print(f"   ✅ Scaled using: {scaler_name}")
    except Exception as e:
        print(f"   ⚠️ Scaling failed: {e}")
        X_test_scaled = X_test
else:
    X_test_scaled = X_test
    print("   ⚠️ No scaler found, using raw data")

# ============================================================
# 5. MAKE PREDICTIONS
# ============================================================

print("\n" + "=" * 80)
print("  🔮 MAKING PREDICTIONS")
print("=" * 80)

predictions = {}

for name, model in models.items():
    try:
        print(f"\n🧠 Predicting with: {name}")
        
        # Check if it's a PyTorch model (state dict)
        if isinstance(model, dict):
            print("   ⚠️ This is a state dict, not a full model. Skipping prediction.")
            continue
        
        # Try to predict
        if hasattr(model, 'predict'):
            pred = model.predict(X_test_scaled)
        elif hasattr(model, 'forward'):
            import torch
            with torch.no_grad():
                X_tensor = torch.FloatTensor(X_test_scaled)
                if torch.cuda.is_available():
                    X_tensor = X_tensor.cuda()
                    model = model.cuda()
                pred = model(X_tensor).cpu().numpy()
                if len(pred.shape) > 1 and pred.shape[1] == 1:
                    pred = pred.flatten()
        else:
            print(f"   ⚠️ Unknown model type, skipping")
            continue
        
        predictions[name] = pred
        print(f"   ✅ Predictions shape: {pred.shape}")
        print(f"   📊 Sample predictions: {pred[:5].round(2)}")
        
    except Exception as e:
        print(f"   ❌ Prediction failed: {e}")

if not predictions:
    print("\n❌ No predictions were made. Please check your models.")
    sys.exit(1)

# ============================================================
# 6. EVALUATE PREDICTIONS
# ============================================================

print("\n" + "=" * 80)
print("  📊 EVALUATION RESULTS")
print("=" * 80)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compute_metrics(y_true, y_pred):
    """Compute all metrics"""
    
    # Ensure same length
    min_len = min(len(y_true), len(y_pred))
    y_true = np.array(y_true[:min_len])
    y_pred = np.array(y_pred[:min_len])
    
    # Remove NaN
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    
    if len(y_true) == 0:
        return {'mae': float('nan'), 'rmse': float('nan'), 'r2': float('nan'), 'mape': float('nan')}
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    try:
        r2 = r2_score(y_true, y_pred)
    except:
        r2 = float('nan')
    
    # MAPE
    mask = y_true > 0
    if mask.any():
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = float('nan')
    
    return {
        'mae': float(mae),
        'rmse': float(rmse),
        'r2': float(r2),
        'mape': float(mape),
        'n_samples': len(y_true)
    }

results = {}

for name, pred in predictions.items():
    metrics = compute_metrics(y_test, pred)
    results[name] = metrics
    
    print(f"\n📊 {name.upper()}:")
    print(f"   MAE:  {metrics['mae']:.4f}")
    print(f"   RMSE: {metrics['rmse']:.4f}")
    print(f"   R²:   {metrics['r2']:.4f}")
    print(f"   MAPE: {metrics['mape']:.2f}%")
    print(f"   Samples: {metrics['n_samples']}")

# ============================================================
# 7. FIND BEST MODEL
# ============================================================

print("\n" + "=" * 80)
print("  🏆 BEST MODEL")
print("=" * 80)

# Find best model by R²
best_model = None
best_r2 = -float('inf')

for name, metrics in results.items():
    r2 = metrics.get('r2', -float('inf'))
    if r2 > best_r2:
        best_r2 = r2
        best_model = name

if best_model:
    print(f"\n✅ Best Model: {best_model.upper()}")
    print(f"   R² Score: {best_r2:.4f}")
    print(f"   MAE: {results[best_model]['mae']:.4f}")
    print(f"   RMSE: {results[best_model]['rmse']:.4f}")
    print(f"   MAPE: {results[best_model]['mape']:.2f}%")

# ============================================================
# 8. VISUALIZATION (Optional)
# ============================================================

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print("\n" + "=" * 80)
    print("  📈 VISUALIZATION")
    print("=" * 80)
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Actual vs Predicted for best model
    ax = axes[0, 0]
    if best_model in predictions:
        ax.scatter(y_test, predictions[best_model], alpha=0.5, s=20)
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
        ax.set_xlabel('Actual RUL')
        ax.set_ylabel('Predicted RUL')
        ax.set_title(f'Actual vs Predicted - {best_model.upper()}')
        ax.grid(True, alpha=0.3)
    
    # Plot 2: Residuals
    ax = axes[0, 1]
    if best_model in predictions:
        residuals = y_test - predictions[best_model]
        ax.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
        ax.axvline(0, color='r', linestyle='--', linewidth=2)
        ax.set_xlabel('Residual')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Residual Distribution - {best_model.upper()}')
        ax.grid(True, alpha=0.3)
    
    # Plot 3: Model Comparison (Bar Chart)
    ax = axes[1, 0]
    model_names = list(results.keys())
    r2_values = [results[m]['r2'] for m in model_names]
    colors = ['#00b4aa' if m == best_model else '#94a3b8' for m in model_names]
    bars = ax.bar(model_names, r2_values, color=colors)
    ax.set_xlabel('Model')
    ax.set_ylabel('R² Score')
    ax.set_title('Model Comparison (R²)')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, r2_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    
    # Plot 4: MAE Comparison
    ax = axes[1, 1]
    mae_values = [results[m]['mae'] for m in model_names]
    bars = ax.bar(model_names, mae_values, color='#3b82f6')
    ax.set_xlabel('Model')
    ax.set_ylabel('MAE')
    ax.set_title('Model Comparison (MAE)')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, mae_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = EXTRACT_DIR / 'evaluation_results.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"\n✅ Figure saved to: {fig_path}")
    
    plt.show()
    
except ImportError:
    print("\n⚠️ Matplotlib not available for visualization")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# 9. SAVE RESULTS
# ============================================================

print("\n" + "=" * 80)
print("  💾 SAVING RESULTS")
print("=" * 80)

# Save results to JSON
results_path = EXTRACT_DIR / 'evaluation_results.json'
with open(results_path, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'results': results,
        'best_model': best_model,
        'best_r2': best_r2,
        'n_samples': len(y_test),
        'n_features': X_test.shape[1]
    }, f, indent=2)

print(f"✅ Results saved to: {results_path}")

# Save predictions
predictions_df = pd.DataFrame()
predictions_df['Actual'] = y_test[:len(next(iter(predictions.values())))]
for name, pred in predictions.items():
    predictions_df[name.upper()] = pred[:len(predictions_df)]

pred_path = EXTRACT_DIR / 'predictions.csv'
predictions_df.to_csv(pred_path, index=False)
print(f"✅ Predictions saved to: {pred_path}")

# ============================================================
# 10. SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("  ✅ EVALUATION COMPLETE")
print("=" * 80)

print(f"\n📊 Summary:")
print(f"   Total Models Evaluated: {len(results)}")
print(f"   Best Model: {best_model.upper() if best_model else 'N/A'}")
print(f"   Best R²: {best_r2:.4f}" if best_model else "")
print(f"   Results Saved: {results_path}")
print(f"   Predictions Saved: {pred_path}")

print("\n📁 Files in extracted directory:")
for file in EXTRACT_DIR.glob('*'):
    size = file.stat().st_size / 1024  # KB
    print(f"   - {file.name} ({size:.1f} KB)")

print("\n" + "=" * 80)

  SYNAPSEPM - REAL TRAINED MODEL EVALUATION
  Started: 2026-08-16 16:39:55

  STEP 1 - LOADING DATASET
✅ Dataset: C:\Users\DELL\OneDrive\Desktop\Synapse PM\data\equipment_anomaly_data.csv
✅ Rows: 7,672
✅ Columns: 7
   ['temperature', 'pressure', 'vibration', 'humidity', 'equipment', 'location', 'faulty']

  STEP 2 - TARGET DETECTION
🎯 Target: faulty

  STEP 3 - FEATURE DETECTION
📊 Numeric features detected: 4
    1. temperature
    2. pressure
    3. vibration
    4. humidity

  STEP 4 - DATA PREPARATION
✅ X shape: (7672, 4)
✅ y shape: (7672,)
🎯 Target min:  0.000000
🎯 Target max:  1.000000
🎯 Target mean: 0.099974
🎯 Target std:  0.299965
🎯 Unique target values: 2
   Values: [0. 1.]

  STEP 5 - LOADING TRAINED MODEL ZIP
✅ Found: trained_model.zip

  STEP 6 - EXTRACTING TRAINED MODEL
✅ Extracted to: C:\Users\DELL\OneDrive\Desktop\Synapse PM\extracted_trained_model

  STEP 7 - MODEL PACKAGE CONTENTS
   ensembles\datasets_aps_failure_test_set_ensemble\ensemble.pkl  (17.48 MB)
   ensembles\

SystemExit: 2